# Experiment 1 & 3: MEDAL vs. AB-MEDAL on GSM8K — Closed-Setup Test Suite ($B = 512$, 4 Configurations)

This self-contained notebook runs the locked **Experiment 1 (E1)** (Accuracy and Inference Efficiency at Matched Search Budgets) and **Experiment 3 (E3)** (Explanatory Compute Allocation and Tree Topology Analysis) on the predetermined **GSM8K `test` split** using `LLaDA-8B-Instruct` with **cooperative varlen FlashAttention-2 batching**.

It is engineered for a **closed / non-Colab deployment** (local GPU workstation, SLURM/Kubernetes GPU node, SSH server, or Docker container) with a preinstalled environment and local repository. See `docs/carc.md` for the Jupyter/Slurm workflow.

---

## 1. Locked Experimental Protocol (Four Configurations, Zero Test-Set Tuning)
- **Predetermined Test Split (`sample_seed = 1`)**:
  - **Smoke Verification (`NUM_SAMPLES = 2`)**: Pre-configured below to verify all 4 configurations at $B = 512$ on 2 predetermined test examples.
  - **Planned Primary Test Size (`NUM_SAMPLES = 200`)**: Uses the 200 frozen test questions in `configs/gsm8k/experiment_samples.json`.
  - **Extended Test Size (`NUM_SAMPLES = 500`)**: Seamlessly extends to 500 predetermined test questions (`configs/gsm8k/experiment_samples_500.json`, where questions `1..200` are 100% identical to the 200 primary set and `resume=True` continues directly from question `201`).
- **Four Locked Configurations**:
  1. `medal @ B = 256` (Condition A, $L_c = 10$, $C_{\min} = 3$)
  2. `ab_medal @ B = 256` (Condition A, $L_c = 10$, $C_{\min} = 3$)
  3. `medal @ B = 512` (Condition B, $L_c = 10$, $C_{\min} = 3$)
  4. `ab_medal @ B = 512` (Condition B, $L_c = 10$, $C_{\min} = 3$)

| Component | MEDAL | AB-MEDAL |
|---|---|---|
| **Backbone Model** | `GSAI-ML/LLaDA-8B-Instruct` (`bfloat16`, varlen FA2) | Same |
| **Questions** | Predetermined GSM8K `test` (`2` smoke / `200` primary / `500` extended) | Same |
| **Prompt** | Two-shot decomposition (pens & muffins) | Same |
| **Search Budgets** | Continuous search to $B = 512$ with prefix checkpoints at $B \in \{256, 512\}$ | Same |
| **Max Search Depth** | $L_c = 10$ | Same |
| **Eligible Actions** | $K_1 = 3, K_2 = 5$ (Max-Entropy vs. Expected-Entropy) | Same |
| **Immediate Reward** | Normalized IG: $R_{IG}(p, s) = \frac{H(p) - H(s)}{H(p)}$ | Same |
| **Candidate Selection** | Minimum-$C=3$ rule: $d^* = \max\{d \in [1, L_c]: N_d \ge 3\}$ | Same |
| **Completion Decoder** | Batched Deterministic Confidence at $T=0$ (256 length) + Cache | Same |

---

## 2. Continuous Checkpointed Cooperative Batching
Each method runs a single continuous cooperative batched search up to $B = 512$ and captures exact prefix snapshots at $B = 256$ and $B = 512$ ($T_{256}^{(512)} \equiv T_{256}^{(256)}$), sharing physical forwards across active questions. Measure speedup and strict numerical parity on the target GPU; neither is guaranteed by this notebook.

---

## 3. Pure Offline E3 Explanatory Analysis
Immediately after E1 completes, E3 reads the saved checkpoints and traces with **0 additional model forward passes** to explain why or why not AB-MEDAL improves accuracy at a matched search budget.

## 1. Verify GPU Hardware & `BATCH_SIZE` VRAM Capacity
Verify a BF16-capable CUDA GPU. Start with batch size 4 and measure search/completion peaks; host RAM and GPU VRAM are separate resources.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for LLaDA-8B-Instruct inference. Attach a CUDA GPU runtime.")
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / (1024 ** 3)
print(f"GPU: {gpu.name} | Total VRAM: {vram_gib:.2f} GiB")
if vram_gib < 20:
    raise RuntimeError("This notebook uses unquantized BF16 weights. Use a GPU with >= 22 GB VRAM (e.g., L4, RTX 3090/4090, A10G, A100, H100).")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This GPU does not support bfloat16 inference.")

print("GPU check passed. Use the full-pipeline benchmark to measure safe batching; begin at BATCH_SIZE=4.")


## 2. Locate the Existing Repository
Use an existing checkout, optionally specified with `REPO_DIR`. Clone and prepare the environment before launching a CARC job.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"

# Prefer existing local checkout anywhere in cwd or parent directories (closed setup)
local_candidates = [
    Path(os.environ["REPO_DIR"]) if "REPO_DIR" in os.environ else None,
    *[path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/dlm_project").is_dir()],
    Path("/workspace/CSCI544-DLM"),
    Path("/content/CSCI544-DLM"),
]
REPO_DIR = next((p for p in local_candidates if p is not None and (p / "src/dlm_project").is_dir()), Path.cwd())

if not (REPO_DIR / "src/dlm_project").is_dir():
    raise RuntimeError("Set REPO_DIR to an existing checkout; see docs/carc.md.")

os.chdir(REPO_DIR)
for p in (str(REPO_DIR), str(REPO_DIR / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

commit_sha = "unknown"
if (REPO_DIR / ".git").exists():
    res = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    if res.returncode == 0:
        commit_sha = res.stdout.strip()
print(f"✓ Workspace ready (Closed Setup): {REPO_DIR} | commit: {commit_sha}")

## 3. Verify the Prepared Environment
Install dependencies before launching the CARC kernel. This cell validates the selected environment without installing packages.

In [ ]:
import importlib.util
import importlib.metadata
from packaging.requirements import Requirement

required_modules = ["torch", "transformers", "accelerate", "yaml", "matplotlib", "pandas", "seaborn", "pytest"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"Prepare the environment before launching this kernel. Missing: {missing}")
for line in (REPO_DIR / "requirements/llada-colab.txt").read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    requirement = Requirement(line)
    installed = importlib.metadata.version(requirement.name)
    if installed not in requirement.specifier:
        raise RuntimeError(f"{requirement.name} {installed} does not match {requirement.specifier}")
print("Dependencies verified. See docs/carc.md for environment setup.")


## 4. Run Algorithmic Pre-Flight Unit Tests
Run unit tests with synthetic models to confirm checkpoint prefix equivalence, cooperative batch parity (`batch_size=2` vs `batch_size=1`), 200-to-500 predetermined test manifest integrity, completion caching, and E3 offline metric accounting before loading model weights.

In [ ]:
import os
import subprocess
import sys

env = {**os.environ, "PYTHONPATH": f"{REPO_DIR / 'src'}:{REPO_DIR}"}
res = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "tests/decoding/test_checkpoint_equivalence.py",
        "tests/tasks/test_experiment_1.py",
        "tests/tasks/test_e3_analysis.py",
        "-v",
    ],
    cwd=REPO_DIR,
    env=env,
    capture_output=True,
    text=True,
)
print(res.stdout)
if res.returncode != 0:
    print("--- PRE-FLIGHT TESTS FAILED ---")
    print(res.stderr)
    raise RuntimeError("Algorithmic pre-flight tests failed. Review environment before loading weights.")
print("✓ All pre-flight algorithmic and batch-parity unit tests passed.")

## 5. Configure Persistent Storage & Local Cache Paths
Configure local/persistent directories for HuggingFace weights and E1/E3 outputs. Override `PERSIST_ROOT` via environment variable (`export PERSIST_ROOT=/path/to/storage`) if mounting an external volume or NAS.

In [ ]:
from pathlib import Path
import os

PERSIST_ROOT = Path(os.environ.get("PERSIST_ROOT", REPO_DIR / ".cache/e1_test_closed")).resolve()
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HUB_CACHE", str(PERSIST_ROOT / "cache/hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(PERSIST_ROOT / "cache/datasets"))

OUTPUT_DIR = Path(os.environ.get("ARTIFACT_ROOT", PERSIST_ROOT / "results/e1_e3_test")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Persistent Storage Root    :", PERSIST_ROOT)
print("HF Hub Cache Directory     :", os.environ["HF_HUB_CACHE"])
print("Experiment Output Directory:", OUTPUT_DIR)

## 6. Experiment Controls & Locked Test Configuration
Pre-configured for the **2-example closed-setup verification check at $B = 512$** across all 4 configurations (`medal@256`, `ab_medal@256`, `medal@512`, `ab_medal@512`) with cooperative `BATCH_SIZE = 4`.
- To run the **planned primary test set (200 questions)**, set `NUM_SAMPLES = 200`.
- To extend to the **full 500 predetermined test questions**, set `NUM_SAMPLES = 500` (with `resume=True`, it will automatically resume at question 201!).

In [ ]:
import pandas as pd
from IPython.display import display
from dlm_project.tasks.gsm8k.experiment_1 import (
    CHECKPOINT_BUDGETS, MAX_SEARCH_DEPTH, MIN_CANDIDATES, context
)
from scripts.profile_gpu_batch_size import estimate_vram_and_compute

STAGE = "test"
NUM_SAMPLES = 2                     # 2 for smoke check | 200 for primary test | 500 for extended test
METHODS = ["medal", "ab_medal"]     # Both methods evaluated
BUDGETS = [256, 512]                # Continuous search to B=512 with checkpoints at B=256 and B=512 (4 configs)
MAX_DEPTH = MAX_SEARCH_DEPTH        # L_c = 10
MIN_C = MIN_CANDIDATES              # C_min = 3
BATCH_SIZE = 4                      # Cooperative multi-question batch size (set 2, 4, 8, etc. based on GPU VRAM)

ctx = context(
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    num_samples=NUM_SAMPLES,
)

PROFILE_RESOURCES = True

vram_est = estimate_vram_and_compute(batch_sizes=[BATCH_SIZE], max_seq_len=850, max_budget=max(BUDGETS))[0]

print(f"=== Locked Experiment 1 Test Configuration ({STAGE.upper()}) ===")
print("  Protocol Version        :", ctx["protocol_version"])
print(f"  Stage & Question Count  : {ctx['stage']} ({len(ctx['examples'])} predetermined test questions)")
print("  4 Locked Configurations :", [f"{m}@{b}" for b in BUDGETS for m in METHODS])
print("  Max Search Depth (Lc)   :", ctx["max_depth"])
print("  Candidate Selection Rule: min-C =", ctx["num_candidates"])
print(f"  Cooperative Batch Size  : {BATCH_SIZE} (Analytical forward-only estimate, not measured peak: ~{vram_est['est_peak_reserved_gib']:.2f} GiB -> {vram_est['recommended_gpu']})")
print("\nSelected Test Questions Preview:")
for idx, ex in enumerate(ctx["examples"][:min(5, len(ctx["examples"]))]):
    ans = ex.get("reference_solution", "").split("####")[-1].strip()
    print(f"  [{idx+1}] ID: {ex['example_id']:<10} | Ref Answer: {ans:<8} | Q: {ex['question'][:85]}...")

## 7. Cache & Load `LLaDA-8B-Instruct` (With Varlen FlashAttention-2 Patch)
Load `LLaDA-8B-Instruct` in `bfloat16` on CUDA (automatically applying the packed attention patch) and tokenize the test prompts.

In [ ]:
from pathlib import Path
import os
from dlm_project.models.profiles import get_profile
from dlm_project.tasks.gsm8k.experiment_1 import load_model, prepare

profile = get_profile("llada")
model_name = profile["repo_id"].split("/")[-1]

# Check environment override or local staged model directories before falling back to Hub ID
candidate_model_dirs = [
    Path(os.environ["MODEL_PATH"]) if "MODEL_PATH" in os.environ else None,
    PERSIST_ROOT / "models" / model_name,
    REPO_DIR / "models" / model_name,
    Path("/workspace/models") / model_name,
]
local_model_dir = next((p for p in candidate_model_dirs if p is not None and (p / "config.json").is_file()), None)
MODEL_PATH = str(local_model_dir if local_model_dir is not None else profile["repo_id"])
LOCAL_FILES_ONLY = bool(local_model_dir is not None or os.environ.get("HF_HUB_OFFLINE", "") == "1")

print(f"Using model source: {MODEL_PATH} (local_files_only={LOCAL_FILES_ONLY})")
wrapper = load_model(ctx, model_path=MODEL_PATH, device="cuda", local_files_only=LOCAL_FILES_ONLY)
prepared = prepare(wrapper, ctx)

varlen_active = getattr(getattr(wrapper.model, "model", None), "_varlen_patched", False)
print(f"✓ Model loaded: {wrapper.identity['family']} ({wrapper.identity['repo_id']} @ {wrapper.identity['revision']})")
print(f"✓ Varlen FlashAttention-2 Batch Patch Active: {varlen_active}")
print(f"✓ Prepared {len(prepared)} test prompts with two-shot decomposition template.")

## 8. Real-Model CUDA Smoke Test
Verify search execution and checkpoint/NFE counts at small budgets. This is not a serial/batch parity test; use the CARC benchmark for that comparison.

In [ ]:
from dlm_project.tasks.gsm8k.experiment_1 import execute_search_batch_checkpointed

SMOKE_BUDGETS = (2, 4)
print(f"Running fast real-model CUDA smoke check at budgets {SMOKE_BUDGETS} with batch_size={len(prepared[:2])}...")
for smoke_method in METHODS:
    pairs = execute_search_batch_checkpointed(
        wrapper,
        prepared[:2],
        ctx,
        smoke_method,
        budgets=SMOKE_BUDGETS,
        max_depth=MAX_DEPTH,
        num_candidates=MIN_C,
    )
    for item, (tree, elapsed) in zip(prepared[:2], pairs):
        assert 2 in tree.checkpoints and 4 in tree.checkpoints, f"Missing checkpoints for {smoke_method}"
        assert tree.model_calls == 4, f"Expected 4 search NFEs, got {tree.model_calls}"
    print(f"  ✓ {smoke_method.upper():8s}: cooperative batch search & checkpoints {SMOKE_BUDGETS} verified ({pairs[0][1]:.2f}s).")
print("✓ Real-model CUDA pre-flight smoke check passed!")

## 9. Execute Locked Experiment 1 Test Run ($B = 512$, All 4 Configurations)
Execute both **MEDAL** and **AB-MEDAL** across the predetermined `test` questions up to $B = 512$ with prefix checkpointing at `BUDGETS = [256, 512]`, cooperative batch size `BATCH_SIZE`, cross-budget/cross-method completion caching, incremental CSV/manifest flushing (`os.sync()`), and automatic checkpoint resumption (`resume=True`).

In [ ]:
from time import perf_counter
from dlm_project.tasks.gsm8k.experiment_1 import run_e1_stage
from scripts.run_e1_test import compute_paired_comparison
from dlm_project.experiments.sweep_report import write_csv

print(
    f"Starting Locked E1 Test Execution on {len(ctx['examples'])} test questions "
    f"for {METHODS} at budgets {BUDGETS} (batch_size={BATCH_SIZE})..."
)
t_start = perf_counter()

q_rows, agg_rows = run_e1_stage(
    wrapper,
    ctx,
    OUTPUT_DIR,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
    max_depth=MAX_DEPTH,
    num_candidates=MIN_C,
    resume=True,
    batch_size=BATCH_SIZE,
    profile_resources=PROFILE_RESOURCES,
)

paired_rows = compute_paired_comparison(q_rows, budgets=BUDGETS)
if paired_rows:
    write_csv(OUTPUT_DIR / "summaries" / f"e1_{STAGE}_paired_comparison.csv", paired_rows)
    write_csv(OUTPUT_DIR / "summaries" / "e1_paired_comparison.csv", paired_rows)

t_total = perf_counter() - t_start
print(f"\n✓ Experiment 1 ({STAGE.upper()}) Completed in {t_total:.1f}s ({t_total/60:.2f} minutes).")
print(f"✓ All 4 configurations ({len(q_rows)} question-budget outcomes) saved to: {OUTPUT_DIR / STAGE}")

## 10. Experiment 1 Results & Paired McNemar Comparison (All 4 Configurations)
Display aggregate accuracy, 95% Wilson confidence intervals, NFE efficiency, paired AB-MEDAL vs. MEDAL accuracy deltas ($\Delta\text{Acc}$ and exact McNemar $p$-value) at matched budgets $B = 256$ and $B = 512$, and per-question outcomes.

In [ ]:
import pandas as pd
from IPython.display import display

agg_file = OUTPUT_DIR / "summaries" / f"e1_{STAGE}_aggregate.csv"
df_agg = pd.read_csv(agg_file)
print("=== Table 1: E1 Aggregate Performance & Efficiency Across All 4 Configurations ===")
display(df_agg[[
    "search_budget", "method", "sample_count", "accuracy_percent",
    "ci_95_low", "ci_95_high", "mean_total_nfe", "mean_selected_depth",
    "mean_search_wall_seconds", "mean_completion_wall_seconds", "mean_total_wall_seconds", "fallback_rate"
]])

paired_file = OUTPUT_DIR / "summaries" / f"e1_{STAGE}_paired_comparison.csv"
if paired_file.is_file():
    df_paired = pd.read_csv(paired_file)
    print("\n=== Table 2: Paired Comparison (AB-MEDAL vs. MEDAL at Matched Search Budgets) ===")
    display(df_paired[[
        "search_budget", "paired_questions", "medal_accuracy", "ab_medal_accuracy",
        "accuracy_delta_percent", "both_correct", "ab_medal_only_correct",
        "medal_only_correct", "both_wrong", "mcnemar_exact_p"
    ]])

pq_file = OUTPUT_DIR / "summaries" / f"e1_{STAGE}_per_question.csv"
df_pq = pd.read_csv(pq_file)
print("\n=== Table 3: Per-Question Outcomes Across All 4 Configurations ===")
display(df_pq[[
    "example_id", "search_budget", "method", "exact_match",
    "predicted_answer", "reference_answer", "selected_depth",
    "search_nfes", "completion_nfes", "total_nfes", "was_cached"
]].head(20))

## 11. Run Offline E3 Explanatory Analysis & Display Publication Figures
Run 100% offline E3 tree-topology, compute-allocation, and reward/redundancy analysis on the saved E1 test checkpoints (**0 extra model forward passes**) to explain why or why not AB-MEDAL improves accuracy at a matched search budget.

In [ ]:
import pandas as pd
from dlm_project.tasks.gsm8k.e3_analysis import analyze_e3_stage
from IPython.display import Image, display, Markdown

offline_started = perf_counter()
print("Running E3 Offline Tree Allocation & Topology Analysis...")
e3_q, e3_agg = analyze_e3_stage(
    results_dir=OUTPUT_DIR / STAGE,
    output_dir=OUTPUT_DIR / "summaries",
    stage=STAGE,
    methods=tuple(METHODS),
    budgets=tuple(BUDGETS),
)
print("✓ E3 Analysis Completed successfully.")

df_e3_agg = pd.DataFrame(e3_agg)
print("\n=== Table 4: E3 Aggregate Structural & Compute Allocation Diagnostics ===")
display(df_e3_agg)

figures_dir = OUTPUT_DIR / "summaries" / "e3_figures"
fig_titles = {
    "fig1_nfe_by_depth": "Figure 1: NFE Allocation by Tree Depth — MEDAL vs AB-MEDAL across budgets",
    "fig2_branching_factors": "Figure 2: Branching Factor Distribution — Children per expanded parent",
    "fig3_candidate_density": "Figure 3: Candidate Density vs Depth — Minimum-C availability and selection depth",
    "fig4_root_subtrees": "Figure 4: Root-Subtree Compute Concentration — Search effort across root branches",
    "fig5_reward_redundancy": "Figure 5: Reward and Search Redundancy — Negative IG rates and duplicate states",
    "fig6_accuracy_vs_budget": "Figure 6: Accuracy vs Search Budget — Performance scaling curve",
}

for fig_key, title in fig_titles.items():
    png_path = figures_dir / f"{fig_key}.png"
    if png_path.is_file():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(png_path)))
from dlm_project.experiments.resources import finish_reporting
if PROFILE_RESOURCES:
    offline_seconds = perf_counter() - offline_started
    finish_reporting(OUTPUT_DIR, reporting_seconds=max(0.0, offline_started - t_start - t_total),
                     offline_seconds=offline_seconds,
                     total_seconds=perf_counter() - t_start + ctx['setup'].get('model_load_seconds', 0.0))


## 12. Interactive Search Tree & Decision Trace Inspector
Inspect candidate shortlists, selection depth gap $\Delta_d = d_{\max} - d^*$, and AB-MEDAL `GEN`/`CONT` posterior decision traces at $B = 256$ or $B = 512$ for any test question.

In [ ]:
import json
from pathlib import Path

INSPECT_QUESTION_IDX = 0
INSPECT_BUDGET = 512  # Inspect B=512 (or set to 256)

q_info = ctx["examples"][INSPECT_QUESTION_IDX]
ex_id = q_info["example_id"]
print(f"=== Inspecting Test Question {ex_id} (Budget B={INSPECT_BUDGET}) ===")
print("Question:\n", q_info["question"])
print("Reference Answer:", q_info.get("reference_solution", "").split("####")[-1].strip())
print(f"Results Directory: {OUTPUT_DIR / STAGE}")

for m in METHODS:
    chk_dir = OUTPUT_DIR / STAGE / m / ex_id / "checkpoints" / str(INSPECT_BUDGET)
    sel_path = chk_dir / "selection.json"
    met_path = chk_dir / "metrics.json"
    comp_path = chk_dir / "completion.json"

    if sel_path.is_file() and met_path.is_file():
        sel_record = json.loads(sel_path.read_text())
        met_record = json.loads(met_path.read_text())
        comp_record = json.loads(comp_path.read_text()) if comp_path.is_file() else {}

        print(f"\n--- {m.upper()} @ B={INSPECT_BUDGET} ---")
        print(f"  Predicted Answer : {met_record.get('predicted_answer')} (Exact Match: {met_record.get('exact_match')})")
        print(f"  Selected Node ID : {sel_record.get('selected_node_id')} | Selected Depth: {sel_record.get('selected_depth')}")
        print(f"  Selected Q-Value : {met_record.get('selected_q', 0.0):.4f}")
        print(f"  Max Depth Reached: {met_record.get('max_depth_reached')} | Selection Gap (Δd): {met_record.get('selection_depth_gap')}")
        print(f"  Distinct by Depth: {met_record.get('distinct_candidates_by_depth')}")
        print(f"  Shortlist IDs    : {sel_record.get('shortlist_ids')}")
        print(f"  Completion Cached: {comp_record.get('was_cached')} | Completion NFEs: {comp_record.get('completion_nfes')}")
    else:
        print(f"\n--- {m.upper()} @ B={INSPECT_BUDGET} ---")
        print(f"  [Not Found] Expected checkpoint files at: {chk_dir}")

dec_path = OUTPUT_DIR / STAGE / "ab_medal" / ex_id / "decisions.jsonl"
if dec_path.is_file():
    dec_events = [json.loads(line) for line in dec_path.read_text().splitlines()]
    print(f"\n--- AB-MEDAL Decision Trace Sample (first 10 of {len(dec_events)} decisions) ---")
    for d in dec_events[:10]:
        print(
            f"  Cycle {d.get('cycle')}: node {d.get('node_id')} -> "
            f"{d.get('choice', d.get('decision_type'))} (forced={d.get('forced')})"
        )

## 13. Closed-Setup Artifact Sync & Portable Archive Export
Flush OS filesystem buffers and optionally package `summaries/` (CSVs, manifest, and 6 publication figures) into a portable `.tar.gz` archive for easy download from a remote GPU server or cluster.

In [ ]:
import os
import tarfile

try:
    os.sync()
except (AttributeError, OSError):
    pass

archive_path = OUTPUT_DIR / f"e1_e3_{STAGE}_n{len(ctx['examples'])}_summaries.tar.gz"
if (OUTPUT_DIR / "summaries").is_dir():
    with tarfile.open(archive_path, "w:gz") as tar:
        tar.add(OUTPUT_DIR / "summaries", arcname="summaries")
        if (OUTPUT_DIR / "run_manifest.json").is_file():
            tar.add(OUTPUT_DIR / "run_manifest.json", arcname="run_manifest.json")

print(f"✓ All artifacts synced and persisted at: {OUTPUT_DIR}")
print(f"✓ Portable summary + figures bundle created at: {archive_path}")